# เทรนโมเดลจำแนกหมาก (ChessGame · วิชา ML)

ขั้นตอน: **Runtime → Change runtime type → T4 GPU** แล้วกด **Runtime → Run all**

1. อัปโหลด `ml_dataset.zip` (สร้างบนคอมด้วย `python -m server.ml.export_dataset`)
2. แบ่ง train / validation **ตามภาพกระดาน** (ไม่ใช่สุ่มทีละช่อง กันความแม่นสูงเกินจริง)
3. Baseline: Color histogram + SVM
4. CNN ขนาดเล็ก + data augmentation
5. เทียบผล, confusion matrix
6. Export เป็น ONNX แล้วดาวน์โหลด `piece_classifier.onnx` + `labels.json`
   ไปวางที่ `server/data/models/` แล้วกด "โหลดโมเดลใหม่" ในหน้า /board

In [ ]:
# ===== ตั้งค่า =====
LEVEL = 3          # 3 = ว่าง/ขาว/ดำ (เริ่มที่นี่) · 13 = แยกชนิดหมากด้วย
EPOCHS = 25
BATCH = 128
SEED = 42

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()                      # เลือกไฟล์ ml_dataset.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall('data')
print(os.listdir('data'))

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('data/manifest.csv')
df['label'] = df['label3'] if LEVEL == 3 else df['label13']
CLASSES = (['empty', 'white', 'black'] if LEVEL == 3 else
           ['empty'] + [f'{c}_{p}' for c in ('white', 'black')
                        for p in ('pawn', 'knight', 'bishop', 'rook', 'queen', 'king')])
df['y'] = df['label'].map({c: i for i, c in enumerate(CLASSES)})

# แบ่งตามภาพกระดาน (board_id): ช่องจากภาพเดียวกันอยู่ชุดเดียวกันเสมอ
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(split.split(df, groups=df['board_id']))
train_df, val_df = df.iloc[train_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)
print('จำนวนภาพกระดาน:', df.board_id.nunique(),
      '| train', train_df.board_id.nunique(), 'ภาพ', len(train_df), 'ช่อง',
      '| val', val_df.board_id.nunique(), 'ภาพ', len(val_df), 'ช่อง')
print(df['label'].value_counts())
if df.board_id.nunique() < 10:
    print('⚠️ ภาพกระดานน้อยเกินไป ผลที่วัดได้ยังเชื่อถือไม่ได้ ควรเก็บอย่างน้อย 30 ภาพ')

## Baseline: Color histogram + SVM
ใช้ฮิสโตแกรมสี HSV (เฉด/ความอิ่มตัว/ความสว่าง) ของแต่ละช่องเป็น feature
เป็นวิธี ML แบบดั้งเดิม ใช้เทียบกับ CNN ในรายงาน

In [ ]:
import cv2
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def hist_feature(path):
    img = cv2.imread('data/' + path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    feats = [cv2.calcHist([hsv], [c], None, [16], [0, 256 if c else 180]).ravel() for c in range(3)]
    f = np.concatenate(feats)
    return f / (f.sum() + 1e-6)

Xtr = np.stack([hist_feature(p) for p in train_df.file]); ytr = train_df.y.values
Xva = np.stack([hist_feature(p) for p in val_df.file]);   yva = val_df.y.values
svm = SVC(kernel='rbf', C=10, class_weight='balanced').fit(Xtr, ytr)
svm_pred = svm.predict(Xva)
print('SVM accuracy:', round(accuracy_score(yva, svm_pred), 4))
print(classification_report(yva, svm_pred, labels=range(len(CLASSES)), target_names=CLASSES, zero_division=0))

## CNN
ภาพเข้า 96x64 (ช่องรวมส่วนหัวหมากที่ล้ำขึ้นไปช่องบน) · 3 ชั้น Conv → Global Average Pooling → Linear
Data augmentation: ปรับแสง/คอนทราสต์ เลื่อนภาพเล็กน้อย กลับซ้ายขวา (จำลองแสงและการ calibrate ที่คลาดเล็กน้อย)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

train_tf = transforms.Compose([
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.03),
    transforms.RandomAffine(degrees=4, translate=(0.06, 0.04), scale=(0.93, 1.07)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),             # RGB ค่า 0–1 (ตรงกับฝั่งเซิร์ฟเวอร์: blobFromImages scale 1/255, swapRB)
])
val_tf = transforms.ToTensor()

class Squares(Dataset):
    def __init__(self, frame, tf): self.frame, self.tf = frame, tf
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        return self.tf(Image.open('data/' + row.file).convert('RGB')), int(row.y)

train_dl = DataLoader(Squares(train_df, train_tf), batch_size=BATCH, shuffle=True, num_workers=2)
val_dl = DataLoader(Squares(val_df, val_tf), batch_size=BATCH, num_workers=2)

def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.MaxPool2d(2))

class PieceNet(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.features = nn.Sequential(block(3, 16), block(16, 32), block(32, 64))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, n))
    def forward(self, x):
        return self.head(self.features((x - 0.5) / 0.5))   # normalize อยู่ในโมเดล

model = PieceNet(len(CLASSES)).to(device)
# ถ่วงน้ำหนักคลาส: ช่องว่างมีมากกว่าหมากหลายเท่า
counts = np.bincount(train_df.y, minlength=len(CLASSES)).astype(float)
weights = torch.tensor(counts.sum() / (len(CLASSES) * np.maximum(counts, 1)), dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=weights)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
print('parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
history = {'train_loss': [], 'val_acc': []}
best_acc, best_state = 0.0, None

def evaluate():
    model.eval(); preds, probs = [], []
    with torch.no_grad():
        for x, _ in val_dl:
            p = torch.softmax(model(x.to(device)), 1).cpu()
            probs.append(p); preds.append(p.argmax(1))
    return torch.cat(preds).numpy(), torch.cat(probs).numpy()

for epoch in range(EPOCHS):
    model.train(); total = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
        total += loss.item() * len(x)
    sched.step()
    pred, _ = evaluate()
    acc = accuracy_score(val_df.y, pred)
    history['train_loss'].append(total / len(train_df)); history['val_acc'].append(acc)
    if acc >= best_acc:
        best_acc, best_state = acc, {k: v.clone() for k, v in model.state_dict().items()}
    print(f'epoch {epoch + 1:2d}  loss {history["train_loss"][-1]:.4f}  val acc {acc:.4f}')

model.load_state_dict(best_state)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history['train_loss']); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(history['val_acc']); ax[1].set_title('validation accuracy'); ax[1].set_xlabel('epoch')
plt.show()

## ผลเทียบ SVM กับ CNN

In [ ]:
cnn_pred, cnn_prob = evaluate()
print(f'SVM (color histogram): {accuracy_score(yva, svm_pred):.4f}')
print(f'CNN                  : {accuracy_score(yva, cnn_pred):.4f}')
print(classification_report(yva, cnn_pred, labels=range(len(CLASSES)), target_names=CLASSES, zero_division=0))

# ความแม่นระดับ "ทั้งกระดาน": ทายถูกครบ 64 ช่องกี่ภาพ
val_df['ok'] = (cnn_pred == yva)
boards_ok = val_df.groupby('board_id').ok.all()
print(f'กระดานที่ทายถูกครบ 64 ช่อง: {boards_ok.sum()}/{len(boards_ok)} ({boards_ok.mean():.1%})')

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, pred, title in ((ax[0], svm_pred, 'SVM'), (ax[1], cnn_pred, 'CNN')):
    ConfusionMatrixDisplay.from_predictions(yva, pred, labels=range(len(CLASSES)), display_labels=CLASSES,
                                            ax=a, colorbar=False, xticks_rotation=45)
    a.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
# ดูภาพที่ CNN ทายผิด (ช่วยหาสาเหตุ เช่น หมากล้ำช่อง แสงสะท้อน calibrate เพี้ยน)
wrong = np.where(cnn_pred != yva)[0][:24]
if len(wrong):
    fig, axes = plt.subplots(3, 8, figsize=(14, 7))
    for a in axes.ravel(): a.axis('off')
    for a, i in zip(axes.ravel(), wrong):
        a.imshow(Image.open('data/' + val_df.file[i]))
        a.set_title(f'{val_df.square[i]}\nจริง {CLASSES[yva[i]]}\nทาย {CLASSES[cnn_pred[i]]}', fontsize=7)
    plt.tight_layout(); plt.show()
else:
    print('ไม่มีช่องที่ทายผิด')

## Export เป็น ONNX
ฝั่งเซิร์ฟเวอร์ใช้ `cv2.dnn` อ่านโมเดล ส่งภาพทีละ 64 ช่องพร้อมกัน จึงกำหนด batch = 64 ตายตัว
และใส่ softmax ไว้ในโมเดล ให้ผลออกมาเป็นความน่าจะเป็นเลย

In [ ]:
import json
class Exported(nn.Module):
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, x): return torch.softmax(self.net(x), dim=1)

export_model = Exported(model.cpu().eval())
dummy = torch.zeros(64, 3, 96, 64)
kwargs = dict(input_names=['input'], output_names=['probs'], opset_version=13)
try:
    torch.onnx.export(export_model, dummy, 'piece_classifier.onnx', dynamo=False, **kwargs)
except TypeError:   # torch รุ่นเก่าไม่มีพารามิเตอร์ dynamo
    torch.onnx.export(export_model, dummy, 'piece_classifier.onnx', **kwargs)
json.dump({'classes': CLASSES}, open('labels.json', 'w'))

# ตรวจว่า OpenCV อ่านโมเดลได้และได้ผลเท่ากับ PyTorch (ฝั่งเซิร์ฟเวอร์ใช้วิธีเดียวกันนี้)
net = cv2.dnn.readNetFromONNX('piece_classifier.onnx')
imgs = [cv2.imread('data/' + f) for f in val_df.file[:64]]
imgs += [imgs[0]] * (64 - len(imgs))
net.setInput(cv2.dnn.blobFromImages(imgs, 1 / 255, (64, 96), swapRB=True))
cv_probs = net.forward()
torch_probs = export_model(torch.stack([val_tf(Image.open('data/' + f).convert('RGB')) for f in val_df.file[:64]] +
                                       [val_tf(Image.open('data/' + val_df.file[0]).convert('RGB'))] * (64 - min(64, len(val_df))))).detach().numpy()
print('ต่างกันสูงสุด:', float(np.abs(cv_probs - torch_probs).max()))

files.download('piece_classifier.onnx')
files.download('labels.json')